[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RunFMe/dl-pretraining-crash-course/blob/main/01_how_training_starts/how_training_starts.ipynb)

# How training starts on N GPUs

One fact to take away:

> **A distributed training job is N separate, ordinary processes running the same script.
> They share nothing. Each one knows its place only because the launcher put a few environment variables
> into its environment before starting it.**

Everything else (`init_process_group`, NCCL, `all_reduce`) is built on top of those variables. We will:

1. start N processes and look at what they were given,
2. do the launcher's job by hand with a `for` loop,
3. watch the processes find each other and run their first collective,
4. answer a few questions, and break things on purpose to check the answers.

**How to run.** A notebook kernel is *one* process and the point is *N* processes, so each example is a small script
written to `scripts/` and started from the notebook. With 2 or more NVIDIA GPUs it uses NCCL; on a laptop or on Colab it uses the
`gloo` backend on CPU, and everything still works the same way. On Colab a plain CPU runtime is enough (Colab gives at most one GPU,
and NCCL needs one GPU per process); the whole notebook runs in a few minutes.

In [1]:
import os, re, sys, socket, subprocess
import torch

NGPU = torch.cuda.device_count()
NPROC = NGPU if NGPU >= 2 else 4           # one process per GPU, or 4 CPU processes on a laptop
os.makedirs("scripts", exist_ok=True)
print(f"GPUs: {NGPU} → each example starts {NPROC} processes")

def free_port():
    s = socket.socket(); s.bind(("127.0.0.1", 0)); port = s.getsockname()[1]; s.close(); return port

NOISE = re.compile(r"^(\[[WE]\d|W\d{4} |frame #|Exception raised|\s*$|.*Redirects are currently)")   # PyTorch log chatter

def show(text):
    """Print what the processes said. Tracebacks are shortened to the failing script line + the error."""
    out, in_traceback = [], False
    for line in text.splitlines():
        if NOISE.match(line):
            continue
        if line.startswith("Traceback"):
            in_traceback = True; continue
        if in_traceback:
            if line.startswith("  File") and "scripts/" in line:
                out.append("  " + re.sub(r'File ".*scripts/', 'File "scripts/', line.strip()))
            elif not line.startswith(" "):                      # the exception line itself
                in_traceback = False; out.append("  → " + line)
            continue
        out.append(line)
    print("\n".join(out))

def torchrun(script, nproc=NPROC, env=None):
    """Start scripts/<script> with torchrun, exactly like on a GPU box, and print what the processes say."""
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(nproc),
           "--master_addr", "127.0.0.1", "--master_port", str(free_port()), f"scripts/{script}"]
    print("$ torchrun " + " ".join(cmd[3:]))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "OMP_NUM_THREADS": "1", **(env or {})})
    print(f"  (torchrun itself has pid {p.pid})")
    show(p.communicate(timeout=300)[0])

GPUs: 0 → each example starts 4 processes


## 1. One script, N processes

`torchrun --nproc_per_node=4 train.py` starts your script **4 times**, as 4 independent processes, the way a shell would.
Each one is a full Python interpreter with its own memory, its own imports, later its own copy of the model and its own
CUDA context. Nothing is shared.

So how does process number 3 know that it is number 3, that there are 4 of them, and where the others are?
**It does not know. It is told.** Right before starting each process, torchrun puts a few `KEY=value` pairs into that
process's environment. The script reads them with `os.environ`.

![fig1](https://raw.githubusercontent.com/RunFMe/dl-pretraining-crash-course/main/01_how_training_starts/images/fig1_launch.png)

Here is a script that only reports what it was given:

In [2]:
%%writefile scripts/hello.py
import os, sys

def say(msg): sys.stdout.write(msg + "\n"); sys.stdout.flush()      # one write per line, so N processes don't garble each other

me = {k: os.environ.get(k) for k in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]}
say(f"pid={os.getpid()}  parent={os.getppid()}  {me}")

if me["RANK"] == "0":                      # rank 0 also lists everything else the launcher set
    for k in sorted(os.environ):
        if k.startswith(("GROUP_", "ROLE_", "LOCAL_WORLD", "TORCHELASTIC_", "TORCH_NCCL", "OMP_NUM")):
            say(f"      also: {k}={os.environ[k]}")

Writing scripts/hello.py


In [3]:
torchrun("hello.py")

$ torchrun --nproc_per_node 4 --master_addr 127.0.0.1 --master_port 51148 scripts/hello.py
  (torchrun itself has pid 90797)


pid=90804  parent=90797  {'RANK': '0', 'LOCAL_RANK': '0', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51148'}
      also: GROUP_RANK=0
      also: GROUP_WORLD_SIZE=1
      also: LOCAL_WORLD_SIZE=4
      also: OMP_NUM_THREADS=1
      also: ROLE_NAME=default
      also: ROLE_RANK=0
      also: ROLE_WORLD_SIZE=4
      also: TORCHELASTIC_ERROR_FILE=/var/folders/dr/zhv6s35n59z3wly9b3rjcrx80000gn/T/torchelastic_28pmih5b/none_fji8n60q/attempt_0/0/error.json
      also: TORCHELASTIC_MAX_RESTARTS=0
      also: TORCHELASTIC_RESTART_COUNT=0
      also: TORCHELASTIC_RUN_ID=none
      also: TORCHELASTIC_SIGNALS_TO_HANDLE=SIGTERM,SIGINT,SIGHUP,SIGQUIT
      also: TORCHELASTIC_USE_AGENT_STORE=True
      also: TORCH_NCCL_ASYNC_ERROR_HANDLING=1
pid=90805  parent=90797  {'RANK': '1', 'LOCAL_RANK': '1', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51148'}
pid=90806  parent=90797  {'RANK': '2', 'LOCAL_RANK': '2', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_P

Read the output top to bottom:

* **N different `pid`s, one shared `parent`, and the parent is torchrun.** These are plain child processes; `kill` the parent and they get a `SIGTERM`.
* **The environments are identical except for `RANK` and `LOCAL_RANK`.** Same script, same `WORLD_SIZE`, same `MASTER_ADDR:MASTER_PORT`.
* **Nothing is connected yet.** No socket, no NCCL, no GPU has been touched. The processes have only been told their coordinates.

### The five variables that matter

| Variable | Meaning | Who uses it |
|---|---|---|
| `RANK` | My global index, `0 … WORLD_SIZE-1`. **This is the process's identity.** Two processes running the same script differ only by this number. | `init_process_group` (which slot in the group I take); your code (`if rank == 0: save_checkpoint()`, which shard of the data I read) |
| `WORLD_SIZE` | How many processes are in the job, across all machines. | `init_process_group` (how many to wait for); every collective (how many pieces an `all_reduce` sums) |
| `LOCAL_RANK` | My index *on this machine*. Exists for exactly one purpose: `torch.cuda.set_device(LOCAL_RANK)`, so that process k uses GPU k. PyTorch's distributed code never reads it. | your code only |
| `MASTER_ADDR` | Host name / IP of the **meeting point**: the machine where rank 0 (or the launcher) listens for the others. | `init_process_group` |
| `MASTER_PORT` | Its TCP port. | `init_process_group` |

### The rest torchrun adds (bookkeeping, you rarely touch it)

| Variable | Meaning |
|---|---|
| `LOCAL_WORLD_SIZE` | processes on this machine (`--nproc_per_node`) |
| `GROUP_RANK`, `GROUP_WORLD_SIZE` | index of this machine and number of machines; `0` and `1` on one node. `RANK = GROUP_RANK × LOCAL_WORLD_SIZE + LOCAL_RANK` |
| `ROLE_RANK`, `ROLE_NAME`, `ROLE_WORLD_SIZE` | same as `RANK`/`WORLD_SIZE` within a *role*, for jobs that mix e.g. trainers and evaluators; with one role they equal `RANK`/`WORLD_SIZE` |
| `TORCHELASTIC_RUN_ID` | job id |
| `TORCHELASTIC_RESTART_COUNT`, `TORCHELASTIC_MAX_RESTARTS` | how many times the whole group was restarted after a crash, and the limit (`--max-restarts`) |
| `TORCHELASTIC_USE_AGENT_STORE` | "the launcher already hosts the meeting point at `MASTER_ADDR:MASTER_PORT`, just connect to it" |
| `TORCHELASTIC_ERROR_FILE` | where a crashing process writes its traceback so torchrun can tell you *which rank* died |
| `TORCH_NCCL_ASYNC_ERROR_HANDLING` | make a hung NCCL call crash the process instead of hanging forever |
| `OMP_NUM_THREADS` | set to 1 so that N processes do not each start a thread per CPU core |

## 2. torchrun is just a `for` loop

If the five variables are all a process needs, we can start the processes ourselves. No torchrun, no magic:

In [4]:
BASE_ENV = {k: v for k, v in os.environ.items() if k not in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]}

def start_by_hand(env_per_process, script="hello.py"):
    """Start one copy of scripts/<script> per dict; each process gets exactly those extra variables."""
    procs = []
    for extra in env_per_process:
        print("starting", script, "with", extra)
        procs.append(subprocess.Popen([sys.executable, f"scripts/{script}"], env={**BASE_ENV, "OMP_NUM_THREADS": "1", **extra},
                                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True))
    for p in procs:
        show(p.communicate(timeout=120)[0])

port = free_port()
start_by_hand([{"RANK": str(r), "LOCAL_RANK": str(r), "WORLD_SIZE": "4", "MASTER_ADDR": "127.0.0.1", "MASTER_PORT": str(port)}
               for r in range(4)])

starting hello.py with {'RANK': '0', 'LOCAL_RANK': '0', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
starting hello.py with {'RANK': '1', 'LOCAL_RANK': '1', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
starting hello.py with {'RANK': '2', 'LOCAL_RANK': '2', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
starting hello.py with {'RANK': '3', 'LOCAL_RANK': '3', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
pid=90808  parent=90793  {'RANK': '0', 'LOCAL_RANK': '0', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
      also: OMP_NUM_THREADS=1
pid=90809  parent=90793  {'RANK': '1', 'LOCAL_RANK': '1', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
pid=90810  parent=90793  {'RANK': '2', 'LOCAL_RANK': '2', 'WORLD_SIZE': '4', 'MASTER_ADDR': '127.0.0.1', 'MASTER_PORT': '51153'}
pid=90811  parent=90793  {'RANK': '3', 'LOCAL_RANK': '3', 'WORLD_SIZE': '4'

Same output as before, minus the `also:` lines, and the parent is now the notebook kernel. Slurm's
`srun --ntasks-per-node=8 python train.py` is this loop as well; it just names the variables `SLURM_PROCID`, `SLURM_LOCALID`,
`SLURM_NTASKS` and you copy them into `RANK`, `LOCAL_RANK`, `WORLD_SIZE` yourself.

What torchrun adds on top of the loop: it picks the port, restarts the group when a process dies, and on several machines
it makes the launchers agree on who is machine 0, 1, 2 … That is all.

## 3. The processes find each other: `init_process_group`

Now the real first lines of every training script. Three steps:

1. **Claim my GPU**: `torch.cuda.set_device(LOCAL_RANK)`.
2. **Meet the others**: `dist.init_process_group("nccl")`. This reads `RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`. Rank 0 (or torchrun)
   runs a tiny key→value server over TCP at `MASTER_ADDR:MASTER_PORT`, the *TCPStore*. Every process connects to it and registers as "rank r of W".
   That is the whole rendezvous. **No GPU has talked to any GPU yet.**
3. **The first collective** (here `all_reduce`, in DDP the broadcast of rank 0's weights at construction). This is where NCCL wires the GPUs up:
   it uses the store once more to exchange its own addresses, then connects GPU↔GPU over NVLink/PCIe. From then on every collective is a kernel
   on the GPU and the CPU side only launches it.

![fig2](https://raw.githubusercontent.com/RunFMe/dl-pretraining-crash-course/main/01_how_training_starts/images/fig2_store.png)

In [5]:
%%writefile scripts/train_start.py
"""The first lines of every distributed training script."""
import os, sys, datetime, torch, torch.distributed as dist

def say(msg): sys.stdout.write(msg + "\n"); sys.stdout.flush()      # one write per line, so N processes don't garble each other

rank       = int(os.environ["RANK"])         # who am I              (from the launcher)
world_size = int(os.environ["WORLD_SIZE"])   # how many of us        (from the launcher)
local_rank = int(os.environ["LOCAL_RANK"])   # which GPU is mine     (from the launcher)

if torch.cuda.device_count() >= world_size:
    torch.cuda.set_device(local_rank)                                       # 1. claim my GPU
    device, backend = torch.device("cuda", local_rank), "nccl"
else:
    device, backend = torch.device("cpu"), "gloo"                            # laptop fallback

dist.init_process_group(backend, timeout=datetime.timedelta(seconds=10))    # 2. meet the others at MASTER_ADDR:MASTER_PORT
say(f"[rank {rank}] joined: I am rank {dist.get_rank()} of {dist.get_world_size()}, on {device}")

x = torch.ones(1, device=device) * rank
dist.all_reduce(x)                                                          # 3. first collective: GPUs get connected, then x is summed
say(f"[rank {rank}] all_reduce → {x.item():.0f}   (= sum of the ranks 0..{world_size - 1})")
dist.destroy_process_group()

Writing scripts/train_start.py


In [6]:
torchrun("train_start.py", env={"NCCL_DEBUG": "INFO"})     # on GPUs, NCCL_DEBUG=INFO adds a line when the GPUs get connected

$ torchrun --nproc_per_node 4 --master_addr 127.0.0.1 --master_port 51154 scripts/train_start.py
  (torchrun itself has pid 90812)


[rank 0] joined: I am rank 0 of 4, on cpu
[rank 3] joined: I am rank 3 of 4, on cpu
[rank 2] joined: I am rank 2 of 4, on cpu
[rank 1] joined: I am rank 1 of 4, on cpu
[rank 2] all_reduce → 6   (= sum of the ranks 0..3)
[rank 3] all_reduce → 6   (= sum of the ranks 0..3)
[rank 0] all_reduce → 6   (= sum of the ranks 0..3)
[rank 1] all_reduce → 6   (= sum of the ranks 0..3)


On CPU you see only the two prints per rank. On a GPU machine, between "joined" and "all_reduce →" NCCL prints
`... ncclCommInitRank ... Init COMPLETE` once per rank: that is the moment the GPUs are wired together, *inside the first collective*,
not inside `init_process_group`.

### After this, a training script is the usual loop

```python
model = DistributedDataParallel(model.to(device), device_ids=[local_rank])   # broadcasts rank 0's weights (first collective)
sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)     # RANK decides which shard I read
for batch in DataLoader(dataset, sampler=sampler):
    loss = model(batch).loss; loss.backward()                                 # DDP all_reduces the gradients in backward
    optimizer.step()
if rank == 0: torch.save(model.state_dict(), "ckpt.pt")                       # RANK decides who writes the checkpoint
```

Every process runs this same loop. The only places the processes behave differently are the lines that look at `rank` or `local_rank`.

## 4. Questions

Think first, then run the check cells below. Each cell starts the processes by hand with a deliberately broken environment.

1. **What would happen if we lost the env vars and just ran several copies of `train_start.py` together?** Which line fails, with what error? And if the script had `rank = 0; world_size = 1` hard-coded instead, what would the processes do?
2. **What if we kept the rendezvous address (`MASTER_ADDR`, `MASTER_PORT`) intact and only destroyed the rank variables?** Three cases: `RANK` missing; every process says `RANK=0`; every process says `RANK=1`. Where does each one fail, and what is special about rank 0?
3. Two processes get identical environments except `LOCAL_RANK`. Which lines of `train_start.py` behave differently? What if we swapped the two `LOCAL_RANK`s but kept the `RANK`s?
4. One of four processes was started with a different `MASTER_PORT`. What do the other three experience, and for how long?

In [ ]:
# Q1: no environment variables at all
start_by_hand([{}, {}], script="train_start.py")

In [ ]:
# Q1b: every process hard-wired to "rank 0 of a world of 1" (each needs its own port: each one hosts its own store)
start_by_hand([{"RANK": "0", "LOCAL_RANK": "0", "WORLD_SIZE": "1", "MASTER_ADDR": "127.0.0.1", "MASTER_PORT": str(free_port())} for _ in range(2)],
              script="train_start.py")

In [ ]:
# Q2a: meeting point intact, RANK removed
port = free_port()
start_by_hand([{"LOCAL_RANK": "0", "WORLD_SIZE": "2", "MASTER_ADDR": "127.0.0.1", "MASTER_PORT": str(port)}] * 2, script="train_start.py")

In [ ]:
# Q2b: everyone claims RANK=0
port = free_port()
start_by_hand([{"RANK": "0", "LOCAL_RANK": "0", "WORLD_SIZE": "2", "MASTER_ADDR": "127.0.0.1", "MASTER_PORT": str(port)}] * 2, script="train_start.py")

In [ ]:
# Q2c: everyone claims RANK=1, so nobody is rank 0
port = free_port()
start_by_hand([{"RANK": "1", "LOCAL_RANK": "0", "WORLD_SIZE": "2", "MASTER_ADDR": "127.0.0.1", "MASTER_PORT": str(port)}] * 2, script="train_start.py")

In [ ]:
# Q4: one process with the wrong port
port, wrong = free_port(), free_port()
start_by_hand([{"RANK": str(r), "LOCAL_RANK": str(r), "WORLD_SIZE": "4", "MASTER_ADDR": "127.0.0.1", "MASTER_PORT": str(wrong if r == 3 else port)}
               for r in range(4)], script="train_start.py")

<details>
<summary><b>Answers</b> (click to expand)</summary>

**1.** The processes start fine; the operating system does not care. The first line that asks "who am I", `int(os.environ["RANK"])` (line 6), raises `KeyError: 'RANK'`
(a script that does not read it itself gets the same thing from `init_process_group`: `environment variable RANK expected, but not set`).
Nothing hangs, nothing connects. With `rank = 0; world_size = 1` hard-coded, every process is a complete, valid *single-process* job: it hosts its own store,
`all_reduce` sums one tensor with itself, it trains the whole model on the whole dataset, and nothing is ever synchronised. N times the work, no speed-up, no error.
A missing `RANK` is loud; a wrong one is silent.

**2.** *`RANK` missing:* same `KeyError`, on every process; the address is never even used.
*Everyone `RANK=0`:* rank 0 has a special job: it hosts the store. The first process binds `MASTER_PORT`; the second fails at once with `address already in use`;
the first then waits for `WORLD_SIZE` processes to register and gives up after the timeout (`1/2 clients joined`).
*Everyone `RANK=1`:* nobody hosts, so everyone keeps trying to connect to `MASTER_ADDR:MASTER_PORT` until the timeout.
The address says *where* to meet; the rank says *who* you are at the meeting; and one specific rank also has to open the door.

**3.** Only the lines that use `local_rank`: `torch.cuda.set_device(local_rank)` and therefore which GPU every later `device="cuda"` means.
Swapping the two `LOCAL_RANK`s while keeping `RANK` trains correctly, just on the other GPU; `RANK` decides the data shard, the slot in every collective, and who checkpoints.

**4.** The three with the right port register and wait for the fourth; rank 0 reports `3/4 clients joined` after the timeout (10 s here, 10 minutes by default),
and the others time out waiting for keys that never appear. The odd one out tries to connect to a port where nobody listens and times out too. In real life this is
"the job hangs for 10 minutes and then every rank prints a timeout": always check that every rank got the same `MASTER_ADDR:MASTER_PORT`.

</details>

### Further reading

* Hugging Face Ultra-Scale Playbook, appendix A0 "Parallel programming crash course": the collectives that come after this.
* PyTorch docs: `torchrun` (every variable in section 1) and `torch.distributed` (`init_process_group`, `TCPStore`).